In [1]:
import gc

In [2]:
import numpy as np
import xarray as xr

In [3]:
from data import _load_single_dataset, ds_pressure, ds_single

In [4]:
from dataclasses import dataclass
from enum import StrEnum


class ErrorKind(StrEnum):
    MAX = "max"
    MEAN = "mean"

    def __repr__(self) -> str:
        return f"{type(self).__name__}.{self._name_}"


@dataclass
class ExpertUncertaintyBoundQuantile:
    absolute_value: float
    absolute_percentile: float
    absolute_kind: ErrorKind
    relative_value: float
    relative_percentile: float
    relative_kind: ErrorKind

In [5]:
def compute_uncertainty_bound_quantile(
    mean: xr.DataArray,
    spread: xr.DataArray,
    expert_abs: float,
    expert_abs_kind: ErrorKind,
    expert_rel: float,
    expert_rel_kind: ErrorKind,
) -> ExpertUncertaintyBoundQuantile:
    if np.isnan(expert_abs) and np.isnan(expert_rel):
        return ExpertUncertaintyBoundQuantile(
            absolute=np.nan,
            absolute_value=expert_abs,
            absolute_kind=expert_abs_kind,
            relative=np.nan,
            relative_value=expert_rel,
            relative_kind=expert_rel_kind,
        )

    mean = mean.values.flatten()
    spread = spread.values.flatten()

    spread_nonzero = spread[spread > 0.0]

    if (len(spread_nonzero) > 0) and (not np.isnan(expert_abs)):
        absolute = np.searchsorted(np.sort(spread_nonzero), expert_abs) / len(
            spread_nonzero
        )
    else:
        absolute = np.nan

    abs_mean = np.abs(mean)
    rel = spread[abs_mean > 0.0] / abs_mean[abs_mean > 0.0]
    rel_nonzero = rel[rel > 0.0]

    if (len(rel_nonzero) > 0) and (not np.isnan(expert_rel)):
        relative = np.searchsorted(np.sort(rel_nonzero), expert_rel) / len(rel_nonzero)
    else:
        relative = np.nan

    return ExpertUncertaintyBoundQuantile(
        absolute_value=expert_abs,
        absolute_percentile=absolute,
        absolute_kind=expert_abs_kind,
        relative_value=expert_rel,
        relative_percentile=relative,
        relative_kind=expert_rel_kind,
    )

In [6]:
from tqdm import tqdm

In [7]:
from compression_recommendations import Recommendations
from compression_recommendations.requirements.combinators import (
    AllRequirements,
    AnyRequirement,
)
from compression_recommendations.requirements.error_bounds.max import (
    MaxPointwiseAbsoluteErrorBoundRequirement,
    MaxPointwiseRangeRelativeErrorBoundRequirement,
    MaxPointwiseRelativeErrorBoundRequirement,
)
from compression_recommendations.requirements.error_bounds.mean import (
    MeanAbsoluteErrorBoundRequirement,
    MeanRangeRelativeErrorBoundRequirement,
    MeanRelativeErrorBoundRequirement,
)
from compression_recommendations.requirements.lossless import LosslessRequirement

In [8]:
expert_recommendations = Recommendations.provide

In [9]:
def extract_expert_absolute_relative_bounds(
    da: xr.DataArray,
    markers: dict,
) -> (float, ErrorKind, float, ErrorKind):
    da_range = np.nanmax(da) - np.nanmin(da)

    expert_requirements = expert_recommendations.search(markers=markers)

    # only cover data error bounds in this analysis,
    # skip data limit, etc. requirements
    assert len(expert_requirements) >= 1
    assert isinstance(expert_requirements[0], AnyRequirement | AllRequirements)

    expert_absolute = None
    expert_relative = None

    for expert_requirement in expert_requirements[0].requirements:
        if isinstance(expert_requirement, LosslessRequirement):
            expert_absolute = [ErrorKind.MAX, 0.0]
            continue
        assert isinstance(expert_requirement, AllRequirements)
        expert_kind = None
        for expert_req in expert_requirement.requirements:
            if isinstance(expert_req, MaxPointwiseAbsoluteErrorBoundRequirement):
                assert expert_kind in ["absolute", None]
                expert_kind = "absolute"
                if expert_absolute is None or expert_absolute[1] >= expert_req.value:
                    expert_absolute = [ErrorKind.MAX, expert_req.value]
            elif isinstance(expert_req, MeanAbsoluteErrorBoundRequirement):
                assert expert_kind in ["absolute", None]
                expert_kind = "absolute"
                if expert_absolute is None:
                    expert_absolute = [ErrorKind.MEAN, expert_req.value]
                elif (
                    expert_absolute[0] == ErrorKind.MEAN
                    and expert_absolute[1] > expert_req.value
                ):
                    expert_absolute[1] = expert_req.value
            elif isinstance(expert_req, MaxPointwiseRelativeErrorBoundRequirement):
                assert expert_kind in ["relative", None]
                expert_kind = "relative"
                if expert_relative is None or expert_relative[1] >= expert_req.value:
                    expert_relative = [ErrorKind.MAX, expert_req.value]
            elif isinstance(expert_req, MeanRelativeErrorBoundRequirement):
                assert expert_kind in ["relative", None]
                expert_kind = "relative"
                if expert_relative is None:
                    expert_relative = [ErrorKind.MEAN, expert_req.value]
                elif (
                    expert_relative[0] == ErrorKind.MEAN
                    and expert_relative[1] > expert_req.value
                ):
                    expert_relative[1] = expert_req.value
            elif isinstance(expert_req, MaxPointwiseRangeRelativeErrorBoundRequirement):
                assert expert_kind in ["absolute", None]
                expert_kind = "absolute"
                if expert_absolute is None or expert_absolute[1] >= (
                    expert_req.value * da_range
                ):
                    expert_absolute = [ErrorKind.MAX, expert_req.value * da_range]
            elif isinstance(expert_req, MeanRangeRelativeErrorBoundRequirement):
                assert expert_kind in ["absolute", None]
                expert_kind = "absolute"
                if expert_absolute is None:
                    expert_absolute = [ErrorKind.MEAN, expert_req.value * da_range]
                elif expert_absolute[0] == ErrorKind.MEAN and expert_absolute[1] > (
                    expert_req.value * da_range
                ):
                    expert_absolute[1] = expert_req.value * da_range
            elif isinstance(expert_req, LosslessRequirement):
                assert expert_kind in ["absolute", None]
                expert_kind = "absolute"
                expert_absolute = [ErrorKind.MAX, 0.0]
            else:
                print(expert_req)
                assert False

    if expert_absolute is None:
        expert_absolute = [ErrorKind.MAX, np.nan]
    if expert_relative is None:
        expert_relative = [ErrorKind.MAX, np.nan]

    return (
        expert_absolute[1],
        expert_absolute[0],
        expert_relative[1],
        expert_relative[0],
    )

In [10]:
with open("expert-quantile-single.csv", "w") as f:
    print(
        "name,level,absolute_value,absolute_percentile,absolute_kind,relative_value,relative_percentile,relative_kind",
        flush=True,
        file=f,
    )
    for i, key in tqdm(enumerate(sorted(ds_single.mean))):
        if (i > 0) and (i % 50 == 0):
            del ds_single
            gc.collect()
            ds_single = _load_single_dataset()
        expert_abs, expert_abs_kind, expert_rel, expert_rel_kind = (
            extract_expert_absolute_relative_bounds(
                ds_single.mean[key],
                {
                    "cf-short-name": key,
                    "level-kind": "single",
                },
            )
        )
        quantile = compute_uncertainty_bound_quantile(
            ds_single.mean[key],
            ds_single.spread[key],
            expert_abs,
            expert_abs_kind,
            expert_rel,
            expert_rel_kind,
        )
        print(
            f"{key},single,{quantile.absolute_value},{quantile.absolute_percentile},{quantile.absolute_kind.value},{quantile.relative_value},{quantile.relative_percentile},{quantile.relative_kind.value}",
            flush=True,
            file=f,
        )

226it [25:43,  6.83s/it]


In [11]:
with open("expert-quantile-pressure.csv", "w") as f:
    print(
        "name,level,absolute_value,absolute_percentile,absolute_kind,relative_value,relative_percentile,relative_kind",
        flush=True,
        file=f,
    )
    for key in tqdm(sorted(ds_pressure.mean)):
        for pressure in sorted(ds_pressure.mean[key].pressure_level.values):
            expert_abs, expert_abs_kind, expert_rel, expert_rel_kind = (
                extract_expert_absolute_relative_bounds(
                    ds_pressure.mean[key].sel(pressure_level=pressure),
                    {
                        "cf-short-name": key,
                        "level-kind": "pressure",
                        "level-value": pressure,
                    },
                )
            )
            quantile = compute_uncertainty_bound_quantile(
                ds_pressure.mean[key].sel(pressure_level=pressure),
                ds_pressure.spread[key].sel(pressure_level=pressure),
                expert_abs,
                expert_abs_kind,
                expert_rel,
                expert_rel_kind,
            )
            print(
                f"{key},{pressure}hPa,{quantile.absolute_value},{quantile.absolute_percentile},{quantile.absolute_kind.value},{quantile.relative_value},{quantile.relative_percentile},{quantile.relative_kind.value}",
                flush=True,
                file=f,
            )

100%|█████████████████████████████████████████████████████████████| 16/16 [09:15<00:00, 34.73s/it]
